# Recommender System - BoardGames Dataset

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/processed/boardgames_features.csv")
model = joblib.load("../models/boardgame_rating_predictor.pkl")

Our aim is to build 3 recommender types: similarity based, popularity based (by user preferences) and hybrid (combining the previous)

### Similarity

In [3]:
#There are some columns that won't be used
similarity_features = df.drop(columns=["id","name","average_rating","rank"])

In [4]:
#Scale the data for better performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(similarity_features)

In [5]:
#Similarity matrix
similarity_matrix = cosine_similarity(X_scaled)

#### Simple recommender function

The user should input the name of the game, to find similar ones

In [6]:
def recommend_similar_games(game_name, n=10):

    idx = df[df["name"] == game_name].index[0]                  # find the index of the game

    similarity_scores = list(enumerate(similarity_matrix[idx]))     #similaruty of the game with others (index, score)

    similarity_scores = sorted(                                 #sorted by score
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:n+1]            #remove the game itself (score=1, so it's the first)

    game_indices = [i[0] for i in similarity_scores]        #get the indices of the games

    return df.iloc[game_indices][["name","average_rating"]] 

In [7]:
#Test the function
recommend_similar_games("Carcassonne")

,name,average_rating
3072,Fjords,6.93
767,Kingdom Builder,6.93
3634,Terra Nova,6.45
2872,Gunkimono,6.97
1712,Land vs Sea,7.05
2043,Fjords,6.77
3086,Rheinländer,6.66
4957,Okanagan: Valley of the Lakes,6.57
1363,Domaine,6.98
674,Carcassonne: Hunters and Gatherers,7.13


#### Multiple games recommender function

In [8]:
def similarity_from_favorites(favorite_games):

    indices = df[df["name"].isin(favorite_games)].index

    similarity_scores = similarity_matrix[indices]

    mean_similarity = similarity_scores.mean(axis=0)

    return mean_similarity

In [9]:
#Create a list where the user would store the games
favorite_games = ["Catan", "Dominion", "Carcassonne"]

### Popularity

The user should input the preferences on certain features

In [10]:
#We can try with this features and user inputs:
players = 3
max_time = 120
complexity = 3.5

In [11]:
filtered = df[
    (df["minplayers"] <= players) &
    (df["maxplayers"] >= players) &
    (df["playtime_mean"] <= max_time) &
    (df["averageweight"] <= complexity)
].copy()                                        #to avoid fragmentation in the dataframe

In [12]:
#The features should be the same as the trained model
X = filtered.drop(columns=["id","name","average_rating","rank"])

In [13]:
#Predict
filtered["predicted_rating"] = model.predict(X)

### Hybrid

In [14]:
#calculate similarity and join to the filtered dataframe
similarity_scores = similarity_from_favorites(favorite_games)
filtered["similarity_score"] = similarity_scores[filtered.index]

Normalize scores

In [15]:
# rating goes from 1 to 10, while similarity rates from 0 to 1

In [15]:
#We will use MinMaxScaler
scaler_norm = MinMaxScaler()

filtered["rating_norm"] = scaler_norm.fit_transform(filtered[["predicted_rating"]])
filtered["similarity_norm"] = scaler_norm.fit_transform(filtered[["similarity_score"]])

Combination of the two aspects

In [16]:
filtered["hybrid_score"] = (0.6 * filtered["rating_norm"] + 0.4 * filtered["similarity_norm"])
# We can play with the weights

Final ranking

In [17]:
recommendations = filtered.sort_values("hybrid_score",ascending=False).head(10)
recommendations[["name","predicted_rating","similarity_score","hybrid_score"]]

,name,predicted_rating,similarity_score,hybrid_score
143,Dominion,7.647334,0.383103,0.889894
76,Quacks,7.663973,0.352390,0.869030
130,Dominion: Intrigue,7.511304,0.370361,0.868422
235,Carcassonne,7.264964,0.375654,0.850186
176,Dominion: Second Edition,7.432359,0.353323,0.848941
279,Samurai,7.378819,0.348798,0.840847
604,Catan,7.278071,0.344908,0.828982
121,Inis,7.838098,0.256377,0.814757
14,The Castles of Burgundy,7.859499,0.252917,0.814157
767,Kingdom Builder,7.156688,0.336956,0.812310


## Recommender Function

In [18]:
def recommend_games(players, max_time, complexity, favorite_games, n=10):

    filtered = df[
        (df["minplayers"] <= players) &
        (df["maxplayers"] >= players) &
        (df["playtime_mean"] <= max_time) &
        (df["averageweight"] <= complexity)
    ].copy()

    X = filtered.drop(columns=["id", "name", "average_rating", "rank"])

    filtered["predicted_rating"] = model.predict(X)

    similarity_scores = similarity_from_favorites(favorite_games)

    filtered["similarity_score"] = similarity_scores[filtered.index]

    scaler_norm = MinMaxScaler()

    filtered["rating_norm"] = scaler_norm.fit_transform(filtered[["predicted_rating"]])
    filtered["similarity_norm"] = scaler_norm.fit_transform(filtered[["similarity_score"]])
    filtered["hybrid_score"] = (0.6 * filtered["rating_norm"] + 0.4 * filtered["similarity_norm"])

    return filtered.sort_values("hybrid_score",ascending=False).head(n)[["name","hybrid_score"]]

In [19]:
#test
recommend_games(players=3,max_time=120,complexity=3.5,favorite_games=["Catan","Dominion","Carcassonne"])

,name,hybrid_score
143,Dominion,0.889894
76,Quacks,0.869030
130,Dominion: Intrigue,0.868422
235,Carcassonne,0.850186
176,Dominion: Second Edition,0.848941
279,Samurai,0.840847
604,Catan,0.828982
121,Inis,0.814757
14,The Castles of Burgundy,0.814157
767,Kingdom Builder,0.812310


The recommendator works, but shows the expansions of the main game and the game itself, so we will remove them

In [20]:
def remove_expansions(df, favorite_games):

    mask = pd.Series(False, index=df.index)

    for game in favorite_games:

        mask = mask | df["name"].str.contains(game, case=False, na=False)

    return df[~mask]

## New Recommender Function

Upgrade: new input based on year of the game

In [21]:
def recommend_games(players, max_time, complexity, age, favorite_games, n=10):

    current_year = 2026                     #max year checked in the describe function

    if age == "New":
        year_filter = df["yearpublished"] >= current_year - 5
    elif age == "Classic":
        year_filter = df["yearpublished"] <= current_year - 15
    else:
        year_filter = True                  #not new nor classic


    filtered = df[
        (df["minplayers"] <= players) &
        (df["maxplayers"] >= players) &
        (df["playtime_mean"] <= max_time) &
        (df["averageweight"] <= complexity) &
        year_filter
    ].copy()


    # remove expansions
    mask = pd.Series(False, index=filtered.index)

    for game in favorite_games:

        mask = mask | filtered["name"].str.contains(game,case=False,na=False)

    filtered = filtered[~mask]


    X = filtered.drop(columns=["id", "name", "average_rating", "rank"])

    filtered["predicted_rating"] = model.predict(X)

    similarity_scores = similarity_from_favorites(favorite_games)

    filtered["similarity_score"] = similarity_scores[filtered.index]


    scaler = MinMaxScaler()

    filtered["rating_norm"] = scaler.fit_transform(filtered[["predicted_rating"]])
    filtered["similarity_norm"] = scaler.fit_transform(filtered[["similarity_score"]])
    filtered["hybrid_score"] = (0.6 * filtered["rating_norm"] + 0.4 * filtered["similarity_norm"])


    return filtered.sort_values("hybrid_score",ascending=False).head(n)[["name","hybrid_score"]]

In [22]:
#test
recommend_games(players=3,max_time=120,complexity=3.5,age = "New",favorite_games=["Catan","Dominion","Carcassonne"])

,name,hybrid_score
1712,Land vs Sea,0.859633
998,Cities,0.847130
710,Brian Boru: High King of Ireland,0.843727
238,Castle Combo,0.842717
223,Return to Dark Tower,0.825049
46,Heat: Pedal to the Metal,0.811368
51,Clank!: Catacombs,0.811168
396,Destinies,0.810310
1451,Middle Ages,0.809861
1928,Monkey Palace,0.809758
